## About Dataset
Context
This is a small subset of dataset of Book reviews from Amazon Kindle Store category.

Content
5-core dataset of product reviews from Amazon Kindle Store category from May 1996 - July 2014. Contains total of 982619 entries. Each reviewer has at least 5 reviews and each product has at least 5 reviews in this dataset.
Columns

- asin - ID of the product, like B000FA64PK
- helpful - helpfulness rating of the review - example: 2/3.
- overall - rating of the product.
- reviewText - text of the review (heading).
- reviewTime - time of the review (raw).
- reviewerID - ID of the reviewer, like A3SPTOKDG7WBLN
- reviewerName - name of the reviewer.
- summary - summary of the review (description).
- unixReviewTime - unix timestamp.

Acknowledgements
This dataset is taken from Amazon product data, Julian McAuley, UCSD website. http://jmcauley.ucsd.edu/data/amazon/

License to the data files belong to them.

Inspiration
- Sentiment analysis on reviews.
- Understanding how people rate usefulness of a review/ What factors influence helpfulness of a review.
- Fake reviews/ outliers.
- Best rated product IDs, or similarity between products based on reviews alone (not the best idea ikr).
- Any other interesting analysis

#### Best Practises
1. Preprocessing And Cleaning
2. Train Test Split
3. BOW,TFIDF,Word2vec
4. Train ML algorithms

In [1]:
import pandas as pd
df = pd.read_csv('all_kindle_review.csv')

In [2]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [3]:
df = df[['reviewText','rating']]

In [4]:
df.shape

(12000, 2)

In [5]:
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [6]:
df['rating'].unique()

array([3, 5, 4, 2, 1], dtype=int64)

In [7]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [8]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


### Preprocessing and Cleaning

In [9]:
df['rating'] = df['rating'].apply(lambda x: 1 if x>=3 else 0)

In [10]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

1) Lower all the cases

In [11]:
df['reviewText'] = df['reviewText'].str.lower()

In [12]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [13]:
import re
import nltk
from nltk.corpus import stopwords

In [14]:
from bs4 import BeautifulSoup

In [15]:
#Removing Special characters
df['reviewText'] = df['reviewText'].apply(lambda x:re.sub('[^a-z A-Z 0-9-]+',' ',x))
#Remove the stopwords
df['reviewText'] = df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]) )
#Remove the url
df['reviewText'] = df['reviewText'].apply(lambda x:re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
#Remove html tags
df['reviewText'] = df['reviewText'].apply(lambda x: BeautifulSoup(x,'lxml').get_text())
#Remove any additional spaces
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join(x.split()))

In [16]:
# import re
# from bs4 import BeautifulSoup
# from nltk.corpus import stopwords

# stop_words = set(stopwords.words('english'))

# def clean_text(x):
#     # 1) convert to string (handle None, numbers, nan)
#     x = str(x)
    
#     # 2) remove HTML tags safely
#     soup = BeautifulSoup(x, "lxml")
#     for tag in soup(['script', 'style']):
#         tag.decompose()
#     x = soup.get_text(separator=" ", strip=True)
    
#     # 3) remove URLs
#     x = re.sub(r"(http|https|ftp|ssh)://\S+", " ", x)
    
#     # 4) keep only letters, numbers, hyphen, and space
#     x = re.sub(r"[^a-zA-Z0-9\- ]+", " ", x)
    
#     # 5) lowercase
#     x = x.lower()
    
#     # 6) remove stopwords
#     x = " ".join([word for word in x.split() if word not in stop_words])
    
#     # 7) remove extra spaces
#     x = " ".join(x.split())
    
#     return x
# df['reviewText'] = df['reviewText'].apply(clean_text)


In [17]:
from nltk.stem import WordNetLemmatizer

In [18]:
lemmatizer = WordNetLemmatizer()

In [19]:
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

In [20]:
df['reviewText'] = df['reviewText'].apply(lambda x:lemmatize_words(x))

In [26]:
df.head()

,reviewText,rating
0,jace rankin may short nothing mess man hauled ...,1
1,great short read want put read one sitting sex...,1
2,start saying first four book expecting 34 conc...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [28]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['reviewText'],df['rating'],test_size=0.2,random_state=42)

In [29]:
from sklearn.feature_extraction.text import CountVectorizer
bow = CountVectorizer()
X_train = bow.fit_transform(X_train).toarray()
X_test = bow.transform(X_test).toarray()

In [30]:
X_train

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [31]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow = GaussianNB()
nb_model_bow.fit(X_train,y_train)

GaussianNB()

In [32]:
from sklearn.metrics import confusion_matrix,classification_report,accuracy_score

In [33]:
y_pred_bow = nb_model_bow.predict(X_test)

In [34]:
confusion_matrix(y_test,y_pred_bow)

array([[511, 292],
       [749, 848]], dtype=int64)

In [35]:
print("BOW accuracy: ",accuracy_score(y_test,y_pred_bow))

BOW accuracy:  0.56625


TFIDF

In [40]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df['reviewText'],df['rating'],test_size=0.2,random_state=42)

In [41]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer()

In [42]:
X_train = tfidf.fit_transform(X_train).toarray()
X_test = tfidf.transform(X_test).toarray()

In [43]:
X_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [44]:
from sklearn.naive_bayes import GaussianNB
model_tfidf = GaussianNB()
model_tfidf.fit(X_train,y_train)

GaussianNB()

In [45]:
y_pred_tfidf = model_tfidf.predict(X_test)

In [46]:
confusion_matrix(y_test,y_pred_tfidf)

array([[491, 312],
       [713, 884]], dtype=int64)

In [47]:
print("TFIDF accuracy: ",accuracy_score(y_test,y_pred_tfidf))

TFIDF accuracy:  0.5729166666666666
